In [5]:
!pip install mediapipe

  Using cached mediapipe-0.10.21-cp312-cp312-win_amd64.whl.metadata (10 kB)
  Using cached absl_py-2.2.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached jax-0.6.0-py3-none-any.whl.metadata (22 kB)
  Using cached jaxlib-0.6.0-cp312-cp312-win_amd64.whl.metadata (1.2 kB)
  Using cached opencv_contrib_python-4.11.0.86-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached sounddevice-0.5.1-py3-none-win_amd64.whl.metadata (1.4 kB)
  Using cached ml_dtypes-0.5.1-cp312-cp312-win_amd64.whl.metadata (22 kB)
Using cached mediapipe-0.10.21-cp312-cp312-win_amd64.whl (51.0 MB)
Using cached sounddevice-0.5.1-py3-none-win_amd64.whl (363 kB)
Using cached absl_py-2.2.2-py3-none-any.whl (135 kB)
Using cached jax-0.6.0-py3-none-any.whl (2.3 MB)
Using cached jaxlib-0.6.0-cp312-cp312-win_amd64.whl (56.4 MB)
Using cached opencv_contrib_python-4.11.0.86-cp37-abi3-win_amd64.whl (46.2 MB)
Using cached ml_dtypes-0.5.1-cp312-cp312-win_amd64.whl (210 kB)


ERROR: Could not install packages due to an OSError: [WinError 5] 액세스가 거부되었습니다: 'C:\\Users\\user\\anaconda3\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.



In [ ]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os
from glob import glob
from collections import defaultdict


input_root = 'sliced_images'
output_root = 'skeleton_images'
os.makedirs(output_root, exist_ok=True)

#미디어파이프 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)

#관절 및 연결 정의
keypoints = ['Head', 'LShoulder', 'RShoulder', 'LElbow', 'RElbow',
             'LWrist', 'RWrist', 'LHip', 'RHip', 'LKnee', 'RKnee',
             'LAnkle', 'RAnkle', 'Neck', 'Hip']

skeleton_lines = [
    ('Head', 'Neck'),
    ('Neck', 'LShoulder'), ('Neck', 'RShoulder'),
    ('LShoulder', 'LElbow'), ('LElbow', 'LWrist'),
    ('RShoulder', 'RElbow'), ('RElbow', 'RWrist'),
    ('LShoulder', 'LHip'), ('RShoulder', 'RHip'),
    ('LHip', 'LKnee'), ('LKnee', 'LAnkle'),
    ('RHip', 'RKnee'), ('RKnee', 'RAnkle'),
    ('LHip', 'RHip'), ('Hip', 'Neck')
]

#중심 정렬
def center_skeleton(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    points = cv2.findNonZero(255 - gray)
    if points is None:
        return image
    x, y, w, h = cv2.boundingRect(points)
    cx, cy = x + w // 2, y + h // 2
    h_img, w_img = image.shape[:2]
    dx, dy = (w_img // 2 - cx), (h_img // 2 - cy)
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    return cv2.warpAffine(image, M, (w_img, h_img), borderValue=(255, 255, 255))

#actor별 관절 데이터 저장용
actor_pose_data = defaultdict(list)


image_files = glob(os.path.join(input_root, '*', '*', '*', '*.jpg'))
print(f" 총 {len(image_files)}장의 이미지 처리 중...")

for img_path in image_files:
    img_name = os.path.basename(img_path)
    image = cv2.imread(img_path)
    if image is None:
        print(f" 이미지 열기 실패: {img_path}")
        continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)
    landmarks = results.pose_landmarks.landmark if results.pose_landmarks else []

    def get_xy(name):
        try:
            lm = mp_pose.PoseLandmark[name]
            return landmarks[lm].x * image.shape[1], landmarks[lm].y * image.shape[0]
        except:
            return (0, 0)

    def avg_xy(n1, n2):
        x1, y1 = get_xy(n1)
        x2, y2 = get_xy(n2)
        return (x1 + x2) / 2, (y1 + y2) / 2

    parts = img_path.replace('\\', '/').split('/')
    pose_name = parts[-4]     # 자세이름
    actor_folder = parts[-2]  # actorPxxx(n)
    row = {'img_name': img_name}

    if landmarks:
        row.update({
            "Head_x": get_xy("NOSE")[0], "Head_y": get_xy("NOSE")[1],
            "LShoulder_x": get_xy("LEFT_SHOULDER")[0], "LShoulder_y": get_xy("LEFT_SHOULDER")[1],
            "RShoulder_x": get_xy("RIGHT_SHOULDER")[0], "RShoulder_y": get_xy("RIGHT_SHOULDER")[1],
            "LElbow_x": get_xy("LEFT_ELBOW")[0], "LElbow_y": get_xy("LEFT_ELBOW")[1],
            "RElbow_x": get_xy("RIGHT_ELBOW")[0], "RElbow_y": get_xy("RIGHT_ELBOW")[1],
            "LWrist_x": get_xy("LEFT_WRIST")[0], "LWrist_y": get_xy("LEFT_WRIST")[1],
            "RWrist_x": get_xy("RIGHT_WRIST")[0], "RWrist_y": get_xy("RIGHT_WRIST")[1],
            "LHip_x": get_xy("LEFT_HIP")[0], "LHip_y": get_xy("LEFT_HIP")[1],
            "RHip_x": get_xy("RIGHT_HIP")[0], "RHip_y": get_xy("RIGHT_HIP")[1],
            "LKnee_x": get_xy("LEFT_KNEE")[0], "LKnee_y": get_xy("LEFT_KNEE")[1],
            "RKnee_x": get_xy("RIGHT_KNEE")[0], "RKnee_y": get_xy("RIGHT_KNEE")[1],
            "LAnkle_x": get_xy("LEFT_ANKLE")[0], "LAnkle_y": get_xy("LEFT_ANKLE")[1],
            "RAnkle_x": get_xy("RIGHT_ANKLE")[0], "RAnkle_y": get_xy("RIGHT_ANKLE")[1],
            "Neck_x": avg_xy("LEFT_SHOULDER", "RIGHT_SHOULDER")[0],
            "Neck_y": avg_xy("LEFT_SHOULDER", "RIGHT_SHOULDER")[1],
            "Hip_x": avg_xy("LEFT_HIP", "RIGHT_HIP")[0],
            "Hip_y": avg_xy("LEFT_HIP", "RIGHT_HIP")[1]
        })
        actor_pose_data[(pose_name, actor_folder)].append(row)

        #스켈레톤 이미지 생성
        canvas = np.ones((2000, 2000, 3), dtype=np.uint8) * 255
        coords = {k: (int(row[f'{k}_x']), int(row[f'{k}_y'])) for k in keypoints}
        for pt1, pt2 in skeleton_lines:
            cv2.line(canvas, coords[pt1], coords[pt2], (0, 0, 0), 3)
        for x, y in coords.values():
            cv2.circle(canvas, (x, y), 6, (0, 0, 255), -1)

        centered = center_skeleton(canvas)

        output_dir = os.path.join(output_root, pose_name, actor_folder)
        os.makedirs(output_dir, exist_ok=True)
        save_path = os.path.join(output_dir, os.path.splitext(img_name)[0] + '_skeleton.png')
        cv2.imwrite(save_path, centered)
    else:
        print(f" 관절 감지 실패: {img_name}")

#CSV 저장 (actor별)
for (pose_name, actor_folder), rows in actor_pose_data.items():
    output_dir = os.path.join(output_root, pose_name, actor_folder)
    os.makedirs(output_dir, exist_ok=True)
    df = pd.DataFrame(rows)
    csv_path = os.path.join(output_dir, 'pose_keypoints.csv')
    df.to_csv(csv_path, index=False)
    print(f" 저장 완료: {csv_path}")

print(" 전체 스켈레톤 생성 + 중심정렬 + CSV 저장 완료.")


 총 68197장의 이미지 처리 중...
 관절 감지 실패: frame_000042.jpg
 관절 감지 실패: frame_000044.jpg
 관절 감지 실패: frame_000046.jpg
 관절 감지 실패: frame_000047.jpg
 관절 감지 실패: frame_000051.jpg
 관절 감지 실패: frame_000022.jpg
 관절 감지 실패: frame_000025.jpg
 관절 감지 실패: frame_000028.jpg
 관절 감지 실패: frame_000038.jpg
 관절 감지 실패: frame_000028.jpg
 관절 감지 실패: frame_000011.jpg
 관절 감지 실패: frame_000017.jpg
 관절 감지 실패: frame_000147.jpg
 관절 감지 실패: frame_000149.jpg
 관절 감지 실패: frame_000150.jpg
 관절 감지 실패: frame_000025.jpg
 관절 감지 실패: frame_000026.jpg
 관절 감지 실패: frame_000027.jpg
 관절 감지 실패: frame_000169.jpg
 관절 감지 실패: frame_000170.jpg
 관절 감지 실패: frame_000018.jpg
 관절 감지 실패: frame_000161.jpg
 관절 감지 실패: frame_000163.jpg
 관절 감지 실패: frame_000182.jpg
 관절 감지 실패: frame_000024.jpg
 관절 감지 실패: frame_000174.jpg
 관절 감지 실패: frame_000178.jpg
 관절 감지 실패: frame_000185.jpg
 관절 감지 실패: frame_000015.jpg
 관절 감지 실패: frame_000199.jpg
 관절 감지 실패: frame_000000.jpg
 관절 감지 실패: frame_000001.jpg
 관절 감지 실패: frame_000002.jpg
 관절 감지 실패: frame_000003.jpg
 관절 감지 실패: frame_000004.j